# Token and cost analysis - GPT-5-mini
Rebuilds the token and cost ledger only from `results/GPT-5-mini`. Every hypothesis represented by a statistical-analysis artifact in that folder is included. Shared Stage 3 simulator calls are billed once; superseded backup artifacts are excluded. Prices are the fixed article schedule with a 50% flex-tier discount.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'run_analysis': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from aerobat.analysis.costs import (
    cost_summary,
    stage3_agent_summary,
    stage3_round_summary,
    token_calls,
    validate_reported_coverage,
)
from aerobat.utils import save_json

RESULT_ROOTS = [ROOT / 'results' / 'GPT-5-mini']
OUTPUT = ROOT / 'run_analysis' / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
OUTPUT_PREFIX = 'gpt_5_mini'

def _path_label(path):
    path = Path(path)
    return str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path)

def scoped_cost_analysis(result_roots):
    calls = token_calls(result_roots)
    validate_reported_coverage(calls, result_roots)
    stages, overall = cost_summary(calls)
    stage3_agents = stage3_agent_summary(calls)
    stage3_rounds = stage3_round_summary(calls, result_roots)

    stage_values = stages.set_index('accounting_stage')
    agent_values = stage3_agents.set_index('agent')
    numbers = {
        'result_roots': [_path_label(root) for root in result_roots],
        **overall,
        'flex_discount': 0.5,
        'total_uncached_input_tokens': int(calls.uncached_input_tokens.sum()),
        'total_cached_input_tokens': int(calls.cached_input_tokens.sum()),
        'total_output_tokens': int(calls.total_output_tokens.sum()),
        'total_tokens_all_calls': int(calls.total_tokens.sum()),
        'total_default_cost_usd': float(calls.default_cost_usd.sum()),
        'total_flex_cost_usd': float(calls.flex_cost_usd.sum()),
    }

    if 'stage_3' in stage_values.index:
        numbers['stage3_tokens'] = float(stage_values.at['stage_3', 'mean_tokens_per_hypothesis'])
        numbers['stage3_flex_cost_usd'] = float(stage_values.at['stage_3', 'mean_flex_cost_per_hypothesis'])

    for stage_name in ('stage_1', 'stage_2', 'stage_4', 'final_report'):
        if stage_name in stage_values.index:
            numbers[f'{stage_name}_flex_cost_usd'] = float(
                stage_values.at[stage_name, 'mean_flex_cost_per_hypothesis']
            )

    for agent in ('simulator_agent', 'subject_agent', 'research_manager'):
        if agent in agent_values.index:
            numbers[f'stage3_{agent}_flex_cost_usd'] = float(
                agent_values.at[agent, 'mean_flex_cost_per_hypothesis']
            )

    for row in stage3_rounds.itertuples():
        prefix = f'stage3_{row.interaction_rounds}_round'
        numbers[f'{prefix}_hypotheses'] = int(row.hypotheses)
        numbers[f'{prefix}_flex_cost_usd'] = float(row.mean_stage3_flex_cost_usd)

    return calls, stages, stage3_agents, stage3_rounds, numbers

In [ ]:
calls, stages, stage3_agents, stage3_rounds, numbers = scoped_cost_analysis(RESULT_ROOTS)
save_json(numbers, OUTPUT / f'{OUTPUT_PREFIX}_cost_numbers.json')
stages.to_csv(OUTPUT / f'{OUTPUT_PREFIX}_cost_by_stage.csv', index=False)
stage3_agents.to_csv(OUTPUT / f'{OUTPUT_PREFIX}_stage3_cost_by_agent.csv', index=False)
stage3_rounds.to_csv(OUTPUT / f'{OUTPUT_PREFIX}_stage3_cost_by_round_count.csv', index=False)
display(numbers)
display(stages)
display(stage3_agents)
display(stage3_rounds)

## Coverage validation
The call ledger is rebuilt from `results/GPT-5-mini` and validated against analyzed hypotheses in that folder. Shared Stage 3 simulator calls are counted once, and superseded backup artifacts are excluded.

In [ ]:
plot = stages.sort_values('mean_flex_cost_per_hypothesis')
ax = plot.plot.barh(x='accounting_stage', y='mean_flex_cost_per_hypothesis', legend=False, color='#4C78A8')
ax.set(xlabel='Mean flex-tier cost per hypothesis (USD)', ylabel='Stage')
plt.tight_layout()